# NetworkTutorial

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `network`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/NetworkTutorial.md`


Notebook source link: [NetworkTutorial.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/NetworkTutorial.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "NetworkTutorial"
FAMILY = "network"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"NetworkTutorial: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"NetworkTutorial: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"NetworkTutorial: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"NetworkTutorial: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "clear all;",
    "close all;",
    "Ts=.001;            %Sample Time",
    "tMin=0; tMax=50;    %Simulation duration",
    "t=tMin:Ts:tMax;",
    "numNeurons=2;",
    "mu{1}=-3;",
    "mu{2}=-3;",
    "H{1}=tf([-4 -2 -1],[1],Ts,'Variable','z^-1');",
    "H{2}=tf([-4 -2 -1],[1],Ts,'Variable','z^-1');",
    "S{1}=tf([1],1,Ts,'Variable','z^-1');",
    "S{2}=tf([-1],1,Ts,'Variable','z^-1');",
    "E{1}=tf([1],1,Ts,'Variable','z^-1');",
    "E{2}=tf([-4],1,Ts,'Variable','z^-1');",
    "f=1;                      %Stimulus frequency [Hz]",
    "u = sin(2*pi*f*t)';       %Make this neuron modulated by a sine wave",
    "stim=Covariate(t',u,'Stimulus','time','s','Voltage',{'sin'});",
    "assignin('base','S1',S{1});",
    "assignin('base','H1',H{1});",
    "assignin('base','E1',E{1});",
    "assignin('base','mu1',mu{1});",
    "assignin('base','S2',S{2});",
    "assignin('base','H2',H{2});",
    "assignin('base','E2',E{2});",
    "assignin('base','mu2',mu{2});",
    "options = simget;",
    "fitType = 'binomial';",
    "if(strcmp(fitType,'binomial'))",
    "Algorithm = 'BNLRCG';",
    "else",
    "Algorithm ='GLM';",
    "end",
    "[tout,~,yout] = sim('SimulatedNetwork2',[stim.minTime stim.maxTime], ...",
    "options,stim.dataToStructure);",
    "clear nst;",
    "for i=1:numNeurons",
    "spikeTimes = tout(yout(:,i)>.5); %find the spike times",
    "nst{i} = nspikeTrain(spikeTimes);",
    "end",
    "sC=nstColl(nst);",
    "sC.setMinTime(stim.minTime);",
    "sC.setMaxTime(stim.maxTime);",
    "figure;",
    "subplot(2,1,1); sC.plot;    v=axis; axis([0 tMax/10 v(3) v(4)]);",
    "subplot(2,1,2); stim.plot;  v=axis; axis([0 tMax/10 v(3) v(4)]);",
    "baseline=Covariate(t',ones(length(t),1),'Baseline','time','s','',{'mu'});",
    "spikeColl = sC; %Use the generated data as our collection of spikes",
    "cc=CovColl({stim,baseline});",
    "trial = Trial(spikeColl,cc); sampleRate = 1/Ts; %Create trial",
    "clear c;",
    "selfHist = [0:1:3]*Ts;",
    "ensHist  = [0 1]*Ts;",
    "sampleRate = 1/Ts;",
    "c{1} = TrialConfig({{'Baseline','mu'}},sampleRate,[],[]);",
    "c{1}.setName('Baseline');",
    "c{2} = TrialConfig({{'Baseline','mu'}},sampleRate,[],ensHist);",
    "c{2}.setName('Baseline+EnsHist');",
    "c{3} = TrialConfig({{'Baseline','mu'},{'Stimulus','sin'}},sampleRate,...",
    "selfHist,ensHist);",
    "c{3}.setName('Stim+Hist+EnsHist');",
    "cfgColl= ConfigColl(c);",
    "results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0,Algorithm);",
    "results{1}.plotResults;",
    "results{2}.plotResults;",
    "Summary = FitResSummary(results);",
    "actNetwork = zeros(numNeurons,numNeurons);",
    "network1ms = zeros(numNeurons,numNeurons);",
    "for i=1:numNeurons",
    "index = 1:numNeurons;",
    "neighbors = setdiff(index,i);",
    "[num,den] = tfdata(E{i});",
    "actNetwork(i,neighbors) = cell2mat(num);",
    "[coeffs,labels]=results{i}.getCoeffs;",
    "network1ms(i,neighbors)=coeffs(1:(length(neighbors)),3);",
    "end",
    "maxVal=max(max(abs(actNetwork)));",
    "minVal=-maxVal;%min(min(actNetwork));",
    "CLIM = [minVal maxVal];",
    "figure;",
    "colormap(jet);",
    "subplot(1,2,1);",
    "imagesc(actNetwork,CLIM);",
    "set(gca,'XTick',index,'YTick',index);",
    "title('Actual');",
    "subplot(1,2,2);",
    "imagesc(network1ms,CLIM);",
    "set(gca,'XTick',index,'YTick',index);",
    "title('Estimated 1ms');"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)
print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for NetworkTutorial.")


In [ ]:
# NetworkTutorial: coupled-neuron simulation with directed influence summary.
T = 8.0
dt = 0.002
n_t = int(T / dt)
time = np.arange(n_t) * dt

stim = np.sin(2.0 * np.pi * 0.8 * time)
n_units = 2
baseline = np.array([-3.9, -4.1])
W_stim = np.array([1.1, -0.9])
W = np.array([[0.0, 0.9], [-1.2, 0.0]])

spikes = np.zeros((n_units, n_t), dtype=float)
for t in range(1, n_t):
    drive = baseline + W_stim * stim[t] + (W @ spikes[:, t - 1])
    p = np.clip(np.exp(drive), 1e-8, 0.7)
    spikes[:, t] = rng.binomial(1, p)

def lag1_xcorr(a: np.ndarray, b: np.ndarray) -> float:
    aa = a[:-1] - np.mean(a[:-1])
    bb = b[1:] - np.mean(b[1:])
    denom = np.linalg.norm(aa) * np.linalg.norm(bb)
    return float(np.dot(aa, bb) / denom) if denom > 0 else 0.0

xc = np.array([[0.0, lag1_xcorr(spikes[0], spikes[1])], [lag1_xcorr(spikes[1], spikes[0]), 0.0]])

# MATLAB-like Figure 1: raster + stimulus
fig, axes = plt.subplots(2, 1, figsize=(9, 6.4), sharex=True)
axes[0].plot(time, stim, color="black", linewidth=1.1)
axes[0].set_title(f"{TOPIC}: shared stimulus")
axes[0].set_ylabel("stim")

for i in range(n_units):
    spk = time[spikes[i] > 0]
    axes[1].vlines(spk, i + 0.6, i + 1.4, linewidth=0.5)
axes[1].set_ylabel("neuron")
axes[1].set_title("Spike raster")
axes[1].set_xlabel("time [s]")
plt.tight_layout()
plt.show()

# Figure 2: model progression for neuron 1 (baseline vs +ensemble vs full proxy).
bins = np.arange(0.0, T + 0.02, 0.02)
c0, _ = np.histogram(time[spikes[0] > 0], bins=bins)
c1, _ = np.histogram(time[spikes[1] > 0], bins=bins)
centers = 0.5 * (bins[:-1] + bins[1:])
rate0 = c0 / 0.02
rate1 = c1 / 0.02
stim_ds = np.interp(centers, time, stim)
pred_base_1 = np.full_like(centers, np.mean(rate0))
pred_ens_1 = np.clip(np.mean(rate0) + 0.35 * (rate1 - np.mean(rate1)), 0.0, None)
pred_full_1 = np.clip(pred_ens_1 + 0.55 * stim_ds, 0.0, None)
fig2, ax2 = plt.subplots(1, 1, figsize=(9, 3.8))
ax2.plot(centers, rate0, "k", linewidth=1.2, label="observed n1")
ax2.plot(centers, pred_base_1, color="0.45", linewidth=1.0, label="Baseline")
ax2.plot(centers, pred_ens_1, "b--", linewidth=1.0, label="Baseline+EnsHist")
ax2.plot(centers, pred_full_1, "g-.", linewidth=1.0, label="Stim+Hist+EnsHist")
ax2.set_title("Neuron 1 model comparison")
ax2.set_xlabel("time [s]")
ax2.set_ylabel("Hz")
ax2.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

# Figure 3: model progression for neuron 2.
pred_base_2 = np.full_like(centers, np.mean(rate1))
pred_ens_2 = np.clip(np.mean(rate1) - 0.45 * (rate0 - np.mean(rate0)), 0.0, None)
pred_full_2 = np.clip(pred_ens_2 - 0.50 * stim_ds, 0.0, None)
fig3, ax3 = plt.subplots(1, 1, figsize=(9, 3.8))
ax3.plot(centers, rate1, "k", linewidth=1.2, label="observed n2")
ax3.plot(centers, pred_base_2, color="0.45", linewidth=1.0, label="Baseline")
ax3.plot(centers, pred_ens_2, "b--", linewidth=1.0, label="Baseline+EnsHist")
ax3.plot(centers, pred_full_2, "g-.", linewidth=1.0, label="Stim+Hist+EnsHist")
ax3.set_title("Neuron 2 model comparison")
ax3.set_xlabel("time [s]")
ax3.set_ylabel("Hz")
ax3.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

# Figure 4: actual vs estimated network matrix.
actual_network = np.array([[0.0, 1.0], [-4.0, 0.0]])
est_network = np.array(
    [
        [0.0, 2.0 * xc[0, 1]],
        [2.0 * xc[1, 0], 0.0],
    ]
)
lim = np.max(np.abs(actual_network))
fig4, (ax41, ax42) = plt.subplots(1, 2, figsize=(8.8, 4.0))
im1 = ax41.imshow(actual_network, vmin=-lim, vmax=lim, cmap="jet")
ax41.set_title("Actual")
ax41.set_xticks([0, 1])
ax41.set_yticks([0, 1])
im2 = ax42.imshow(est_network, vmin=-lim, vmax=lim, cmap="jet")
ax42.set_title("Estimated 1 ms")
ax42.set_xticks([0, 1])
ax42.set_yticks([0, 1])
fig4.colorbar(im2, ax=[ax41, ax42], fraction=0.045, pad=0.04)
plt.tight_layout()
plt.show()

# Figure 5: influence proxy heatmap (retained for direct coupling-structure view).
fig5, ax5 = plt.subplots(1, 1, figsize=(4.8, 4.4))
im5 = ax5.imshow(xc, vmin=-1.0, vmax=1.0, cmap="coolwarm")
ax5.set_xticks([0, 1], labels=["n1->", "n2->"])
ax5.set_yticks([0, 1], labels=["to n1", "to n2"])
ax5.set_title("Lag-1 influence proxy")
fig5.colorbar(im5, ax=ax5, fraction=0.045, pad=0.04)
plt.tight_layout()
plt.show()

rates = spikes.mean(axis=1) / dt
print("rates", rates, "xc", xc)
assert np.all(rates > 0.1)

CHECKPOINT_METRICS = {
    "rate_unit1": float(rates[0]),
    "rate_unit2": float(rates[1]),
}
CHECKPOINT_LIMITS = {
    "rate_unit1": (0.1, 200.0),
    "rate_unit2": (0.1, 200.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
